# Why Decode is Slow: The Bandwidth Wall

**Workshop Section 2 | ~20 minutes**

LLM inference has two phases: **prefill** (process all input tokens in parallel) and **decode** (generate tokens one at a time). Prefill is compute-bound. Decode is memory-bound. This notebook shows *why* decode hits the bandwidth wall, what the roofline model tells us, and how batching helps (until KV cache runs out of memory).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## The Roofline Model

The roofline model plots achievable performance (FLOP/s) against arithmetic intensity (FLOPs per byte loaded from memory). An operation is either **memory-bound** (left of the ridge point, limited by bandwidth) or **compute-bound** (right of it, limited by peak FLOP/s).

In [ ]:
# A100 80GB specs
peak_flops = 312e12      # 312 TFLOP/s FP16
bandwidth = 2.0e12       # 2 TB/s HBM bandwidth
ridge_point = peak_flops / bandwidth  # FLOPs/byte at ridge

# Arithmetic intensities
ai = np.logspace(-1, 4, 500)
roofline = np.minimum(peak_flops, bandwidth * ai)

# Prefill: ~large matmul, AI ~ 128-256 FLOPs/byte
prefill_ai, prefill_perf = 160, min(peak_flops, bandwidth * 160)
# Decode (batch=1): load full weights for 1 token, AI ~ 1-2 FLOPs/byte
decode_ai, decode_perf = 1.0, min(peak_flops, bandwidth * 1.0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(ai, roofline / 1e12, 'k-', lw=2, label='Roofline (A100 80GB)')
ax.axvline(ridge_point, color='gray', ls='--', alpha=0.5, label=f'Ridge point = {ridge_point:.0f} FLOPs/byte')
ax.plot(prefill_ai, prefill_perf/1e12, 'go', ms=12, label=f'Prefill (AI≈{prefill_ai})')
ax.plot(decode_ai, decode_perf/1e12, 'rs', ms=12, label=f'Decode B=1 (AI≈{decode_ai})')
ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)')
ax.set_ylabel('Performance (TFLOP/s)')
ax.set_title('A100 Roofline: Prefill vs Decode')
ax.legend(loc='lower right')
ax.set_xlim(0.1, 1e4)
ax.set_ylim(0.1, 500)
plt.tight_layout()
plt.show()
print(f'Ridge point: {ridge_point:.0f} FLOPs/byte')
print(f'Decode at batch=1 achieves only {decode_perf/peak_flops*100:.1f}% of peak compute.')

## Why Decode is Memory-Bound

During decode (batch size = 1), generating one token requires a full forward pass through the model. Every weight is loaded from HBM exactly once per token. The arithmetic intensity is:

$$\text{AI}_{\text{decode}} = \frac{2 \times P}{2 \times P} = 1 \text{ FLOP/byte (FP16)}$$

where $P$ = number of parameters. We do ~$2P$ FLOPs (one multiply-add per weight) and load $2P$ bytes (FP16 = 2 bytes/param).

This means **decode speed is entirely determined by memory bandwidth**:

$$\text{tokens/s}_{\text{max}} = \frac{\text{Bandwidth (bytes/s)}}{\text{Model size (bytes)}}$$

No amount of extra compute helps. You are waiting on memory.

In [ ]:
# Model: 7B params in FP16 = 14 GB
model_bytes = 7e9 * 2  # 14 GB

gpus = {
    'A10G (24GB, 600 GB/s)': 600e9,
    'A100 (80GB, 2.0 TB/s)': 2000e9,
    'H100 (80GB, 3.35 TB/s)': 3350e9,
}

print(f'Model: 7B params, FP16 = {model_bytes/1e9:.0f} GB')
print(f'{"="*45}')
for name, bw in gpus.items():
    tok_per_sec = bw / model_bytes
    print(f'{name}: {tok_per_sec:.0f} tokens/s max decode')
print(f'\nThese are theoretical UPPER BOUNDS at batch=1.')
print('Real systems achieve 70-85% of bandwidth limit.')

## Batching Moves You Up

With batch size $B$, we still load each weight once but perform $B$ multiply-adds per weight:

$$\text{AI}_{\text{batched}} = \frac{2PB}{2P} = B \text{ FLOPs/byte}$$

Batching increases arithmetic intensity linearly. At some batch size, we cross the ridge point and become compute-bound. This is why serving systems batch requests together: **throughput scales with batch size until you saturate compute.**

In [ ]:
batch_sizes = np.arange(1, 257)
ai_batched = batch_sizes  # AI = B for decode
perf_batched = np.minimum(peak_flops, bandwidth * ai_batched)
throughput = perf_batched / (2 * 7e9)  # tokens/s total across batch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: AI vs batch size
ax1.plot(batch_sizes, ai_batched, 'b-', lw=2)
ax1.axhline(ridge_point, color='r', ls='--', label=f'Ridge = {ridge_point:.0f}')
cross_b = int(np.ceil(ridge_point))
ax1.axvline(cross_b, color='g', ls=':', alpha=0.7, label=f'Cross at B={cross_b}')
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Arithmetic Intensity (FLOPs/byte)')
ax1.set_title('Batch Size → Arithmetic Intensity')
ax1.legend()

# Right: Throughput vs batch size
ax2.plot(batch_sizes, throughput, 'b-', lw=2)
ax2.axvline(cross_b, color='g', ls=':', alpha=0.7, label=f'Compute-bound at B≥{cross_b}')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Total Throughput (tokens/s)')
ax2.set_title('Decode Throughput vs Batch Size (A100, 7B FP16)')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'To saturate A100 compute, you need batch size ≥ {cross_b}.')
print(f'At B={cross_b}: {throughput[cross_b-1]:.0f} tokens/s total throughput.')

## The Catch: KV Cache Limits Batch Size

Each request in the batch needs its own KV cache. For a 7B model (32 layers, 32 heads, 128 dim/head) at sequence length 2048:

$$\text{KV per request} = 2 \times L \times n_h \times d_h \times \text{seq\_len} \times 2 \text{ bytes}$$

This grows linearly with batch size. At some point, model weights + KV cache > GPU memory, and you **cannot batch further**. This is the fundamental tension: batching helps throughput but KV cache limits how far you can go.

In [ ]:
# 7B model on A100 80GB
gpu_mem = 80e9  # 80 GB
model_mem = 14e9  # 7B FP16
available = gpu_mem - model_mem  # for KV cache + activations
overhead = 2e9  # ~2 GB for activations, CUDA context
kv_budget = available - overhead

# KV cache per request at seq_len=2048
layers, heads, dim_head, seq_len = 32, 32, 128, 2048
kv_per_request = 2 * layers * heads * dim_head * seq_len * 2  # bytes, FP16

max_batch = int(kv_budget / kv_per_request)

print(f'GPU memory:         {gpu_mem/1e9:.0f} GB')
print(f'Model weights:      {model_mem/1e9:.0f} GB')
print(f'Available for KV:   {kv_budget/1e9:.1f} GB')
print(f'KV cache/request:   {kv_per_request/1e6:.0f} MB (seq_len={seq_len})')
print(f'{"="*45}')
print(f'Max batch size:     {max_batch} requests')
print(f'Ridge point needs:  B={cross_b}')
print()
if max_batch >= cross_b:
    print(f'✅ Can reach compute-bound! (max B={max_batch} > ridge={cross_b})')
else:
    print(f'❌ Cannot reach compute-bound (max B={max_batch} < ridge={cross_b})')
print(f'\nAt longer sequences (4096+), KV grows and max batch shrinks.')
print(f'This is why KV cache optimization (GQA, quantization, paging) matters.')